# Lab 1: PINN fundamentals

[Start Here](../../Start_Here.ipynb) · Previous: [Introduction](../../01_Introduction.ipynb) · Next: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb)

This Lab solves the same one-dimensional equation in three ways:

- Forward: learn the solution from the equation and two boundary values.
- Parameterized: add the domain length as a network input.
- Inverse: learn both the solution and an unknown source from observations.

The programs are complete. Run them, then change the settings and compare the results. These Labs are practice, not ranked submissions.

Matching the boundary values alone does not determine the solution. In the inverse run, check the recovered source too: a good-looking solution curve can still have inaccurate second derivatives.

The upstream Lab 1 provides the equations and reference illustrations. This notebook adds executable training code; its network and optimizer settings are not an original upstream training configuration.

<a id="first-problem"></a>
## Lab 1.1: First problem, forward PINN

On $0 \leq x \leq 1$, learn $u(x)$ satisfying $u''(x)=1$ and $u(0)=u(1)=0$. In the general equation below, this first run uses $f(x)=1$.

Read the problem and loss definitions, then run the cells under [Forward PINN execution](#forward-pinn-execution).

### Learn the solution from the equation

A neural network represents the unknown solution. Training reduces its equation residual and boundary errors. Check the result against the analytical solution at independent coordinates; low training loss alone does not establish accuracy everywhere.
For this first problem:
$$
\begin{align} 
    \mathbf{P} : \left\{\begin{matrix}
\frac{\mathrm{d}^2 u}{\mathrm{d} x^2}(x) = f(x), \\ 
\\
u(0) = u(1) = 0,
\end{matrix}\right.
\end{align}
$$
A fully connected network maps one coordinate $x$ to one value $u_{net}(x)$. Smooth activations allow the second derivatives needed by the equation.
Penalize the two boundary values:
$$
\begin{align}
  L_{BC} = u_{net}(0)^2 + u_{net}(1)^2
\end{align}
$$
Use automatic differentiation to evaluate the differential equation at interior coordinates:
$$
\begin{align} 
  L_{residual} = \frac{1}{N}\sum^{N}_{i=1} \left( \frac{\mathrm{d}^2 u_{net}}{\mathrm{d} x^2}(x_i) - f(x_i) \right)^2
\end{align}
$$
The executable forward run minimizes $L = 10L_{BC} + L_{residual}$ on a fixed grid of equation points. L-BFGS updates the network weights in single precision (FP32). For $f(x)=1$, the analytical solution is $\frac{1}{2}(x-1)x$. The figure below is a reference; the training cells will plot your result.
<center><img src="images/single_parabola.png" alt="Drawing" style="width:500px" /></center>

### Defining the equation

```python
class Poisson1D(PDE):
    def __init__(self, inverse=False):
        self.dim = 1
        x = Symbol("x")
        u = Function("u")(x)
        f = Function("f")(x) if inverse else 1
        self.equations = {"poisson": u.diff(x, 2) - f}
```

### Follow one training step

| Operation | Find it in the Python files | Meaning in this problem |
|---|---|---|
| Coordinate to prediction | `BasicPINN.forward` in [pinn_basics.py](source_code/pinn_basics.py) | A batch of $x$ values produces $u_\theta(x)$. |
| Spatial derivatives and residual | `loss_terms` calls `PhysicsInformer` | Differentiate with respect to $x$ to evaluate $u_{\theta,xx}-1$. |
| PDE and boundary losses | `loss_terms` returns `physics` and `boundary` | Penalize interior residuals and $u_\theta(0)^2+u_\theta(1)^2$; forward and parameterized endpoints use soft penalties with weight 10. The inverse example below enforces the endpoints exactly. |
| Parameter gradients and update | `optimize_lab` in [pinn_basics.py](source_code/pinn_basics.py) | `optimizer.step(closure)` calls a function that recomputes the loss and `loss.backward()` on the same fixed training points, then updates the weights and biases. |

Spatial derivatives define the equation residual; parameter gradients determine the weight updates. The solution should be convex and nonpositive between its zero endpoints. Why does the equation alone allow an extra $ax+b$? Which loss term removes that freedom?

<a id="forward-pinn-execution"></a>
### Forward PINN execution

Run the next three cells in order with **Shift+Enter**:

1. Set the device and training steps. On a GPU, confirm that the printed device is `cuda`.
2. Train the forward model.
3. Compare the `analytical` and `PINN` curves.

Check the held-out error in `metrics.json` as well as the plot. The analytical solution is for evaluation, not a training target. A denser evaluation grid checks the whole interval, including its endpoints. Evaluation errors are not used to choose the saved model.

To change the model, edit [pinn_basics.py](source_code/pinn_basics.py), save it, and rerun the training and plotting cells. Changes to a Markdown example do not change the program.

Use 1,000 L-BFGS optimizer calls; there is no Adam stage in this lab. L-BFGS uses recent changes in the gradients to choose an update direction, then tests how far to move along it. One call may evaluate the loss several times, so 1,000 calls does not mean 1,000 loss evaluations. The result panel reports **Lesson accuracy checks: PASS** or **NOT MET**; finishing the run alone does not guarantee an accurate solution.

In [ ]:
import os
import sys
import subprocess
import uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Start_Here.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the bootcamp repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings, completed_output
LAB = ROOT / "01_labs/01_pinn"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs"))).expanduser().resolve()
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "1000"))  # Full-batch L-BFGS optimizer calls
RUN_DIRS = {}
RUN_COMPLETED = {}
validate_settings(DEVICE, STEPS)
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
RUN_COMPLETED["forward"] = False
OUTPUT = OUTPUT_BASE / ("forward-" + uuid.uuid4().hex[:8])
RUN_DIRS["forward"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "forward"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["forward"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "forward")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Lab 1.2: Parameterized problems

How does the solution change when the right endpoint moves? Add the domain length $l\in[1,2]$ as a network input and solve
$$
\begin{align}
    \mathbf{P} : \left\{\begin{matrix}
\frac{\mathrm{d}^2 u}{\mathrm{d} x^2}(x) = f(x), \\ 
\\
u(0) = u(l) = 0,
\end{matrix}\right.
\end{align}
$$
The network now takes both position and length as inputs, $u_{net}(x,l)$.
The L-BFGS run uses a fixed grid: 17 equally spaced lengths in $[1,2]$, with 32 midpoint positions in $[0,l]$ for each length by default. Weight the residual by $l_i$ to account for each interval's length when approximating the domain integral.

$$L_{residual}\approx\frac1N\sum_{i=1}^N l_i\left(u_{xx}(x_i,l_i)-1\right)^2,$$
$$L_{BC}\approx\frac1N\sum_{i=1}^N\left[u(0,l_i)^2+u(l_i,l_i)^2\right].$$

The analytical solution is $u(x,l)=x(x-l)/2$. Fixed evaluation uses $l=1,1.25,1.5,1.75,2$ and reports pooled errors plus `heldout_before/after.per_length`. These five lengths are also in the training grid, so the checks do not establish accuracy at new lengths. Expand the full metrics to compare lengths. The preview and legacy `validation_rmse` show only $l=1.5$.

![Reference illustration of parameterized solutions](images/every_parabola.png)

In [ ]:
RUN_COMPLETED["parameterized"] = False
OUTPUT = OUTPUT_BASE / ("parameterized-" + uuid.uuid4().hex[:8])
RUN_DIRS["parameterized"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "parameterized"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["parameterized"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "parameterized")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

## Lab 1.3: Inverse problems

Now the source $f(x)$ is unknown. Given observations of $u_{true}(x)$ at 100 random points between 0 and 1, train two networks: $u_{net}(x)$ for the solution and $f_{net}(x)$ for the source. They share the equation-residual loss below; only the solution network is fitted to the observations.
$$
\begin{align}
  L_{residual} \approx \left(\int^1_0 dx\right) \frac{1}{N} \sum^{N}_{i=1} \left(\frac{\mathrm{d}^2 u_{net}}{\mathrm{d} x^2}(x_i) - f_{net}(x_i)\right)^2
\end{align}
$$
$$
\begin{align}
  L_{data} = \frac{1}{100} \sum^{100}_{i=1} (u_{net}(x_i) - u_{true}(x_i))^2
\end{align}
$$
Using the function $u_{true}(x)=\frac{1}{48} (8 x (-1 + x^2) - (3 sin(4 \pi x))/\pi^2)$ the solution for $f(x)$ is $x + sin(4 \pi x)$. The figures below are reference illustrations; the execution cells will plot your learned functions.
Reference and learned source $f(x)$:
<center><img src="images/inverse_parabola.png" alt="Drawing" style="width:500px" /><center>

Learned solution $u_{net}(x)$ and observations from $u_{true}$:
<center><img src="images/inverse_parabola_2.png" alt="Drawing" style="width:500px" /><center>

For more examples, see the <a href="https://docs.nvidia.com/physicsnemo/index.html" rel="nofollow">PhysicsNeMo User Documentation</a>.
</center></center></center></center>

### Inverse PINN execution

Train both networks together using the same 100 fixed, noise-free observations of `u` throughout the run. The source values `f` are never training targets.

Write `u(x) = x(1-x) * v(x)` so both zero boundary values hold exactly. Fit the observations in the equivalent form:

```python
v_observed = u_observed / (x_observed * (1 - x_observed))
data_loss = 1000 * mean((v_predicted - v_observed)**2)
```

Only interior observations are used in this division. It weights near-boundary observations more strongly, where small solution errors can hide large curvature errors. The total loss is the PDE residual MSE plus `data_loss`. This scaling is for the supplied noise-free example; it can amplify measurement noise near the boundaries.

Both networks receive smooth coordinate features `[2*x-1, sin(k*pi*x), cos(k*pi*x)]` for `k=1,2,3,4`, followed by two tanh layers of width 32. The solution factor is `v = 0.1 * N_u`; the source is `f = N_f`. The feature set includes frequencies relevant to this example, but no source coefficients or source values are supplied. These are new implementation choices, not recovered upstream settings.

Run 1,000 L-BFGS optimizer calls, then inspect **both** curves. The accuracy check includes `source_rmse` and `source_max_abs` over 401 points, including the endpoints. A small PDE residual alone does not establish correct source recovery.

In [ ]:
RUN_COMPLETED["inverse"] = False
OUTPUT = OUTPUT_BASE / ("inverse-" + uuid.uuid4().hex[:8])
RUN_DIRS["inverse"] = OUTPUT
command = [sys.executable, str(LAB / "source_code/pinn_basics.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + ["--mode", "inverse"]
validate_settings(DEVICE, STEPS)
subprocess.run(command, check=True, cwd=ROOT)
RUN_COMPLETED["inverse"] = True
metrics = show_results(OUTPUT, steps=STEPS, seed=42, preview=False)

In [ ]:
OUTPUT = completed_output(RUN_DIRS, RUN_COMPLETED, "inverse")
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
plt.plot(data["x"], data["reference"], label="analytical")
plt.plot(data["x"], data["prediction"], label="PINN")
plt.xlabel("x"); plt.legend(); plt.show()

In [ ]:
plt.plot(data["x"], data["source_reference"], label="true f(x)")
plt.plot(data["x"], data["source_prediction"], label="inferred f(x)")
plt.legend(); plt.show()

### Experiments and interpretation

- Compare shorter runs with the 1,000-call recipe. Increasing `--steps` alone does not guarantee better accuracy.
- In the inverse problem, vary the number of observations or `INVERSE_DATA_WEIGHT` in the Python file. Compare errors in $u$ and $f$, including the interval endpoints.
- Test additional parameter values and compare their errors; the training loss does not measure performance outside the trained range.

Use `--config` to load the same simple YAML configuration format. The CLI `--steps` argument overrides the YAML steps value.

### Next steps

Before Lab 2, check that you can answer these questions:

- Why does moving the right endpoint from 1 to $L$ give $u=x(x-L)/2$? Check the parameterized prediction and its endpoint values.
- How accurately did the inverse model recover $f(x)$? Compare `source_rmse` with the solution error, not just the plot of $u$.
- Where are the equation and boundary terms in `loss_terms`? Labs 2–4 also provide complete programs. In the Challenges, you will write `student_*` functions in the `.py` files.

Continue to Lab 2 using the link below.

[Start Here](../../Start_Here.ipynb) · Previous: [Introduction](../../01_Introduction.ipynb) · Next: [Lab 2: Projectile motion](../02_projectile/Lab_2_Projectile_Motion.ipynb)

--- 

Further reading: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.